In [ ]:
import os, glob, requests
from bs4 import BeautifulSoup
import faiss
import numpy as np
import google.generativeai as genai
import getpass
import gradio as gr

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("API KEY:")
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])


In [ ]:
textos = []
for filepath in glob.glob("../data/**/*.txt", recursive=True):
    with open(filepath, "r", encoding="utf-8") as f:
        textos.append({"text": f.read(), "source": filepath})

print("Scrapeando datos de la Universidad Nacional...")
res = requests.get("https://repositorio.unal.edu.co/handle/10336/1", verify=False)
if res.status_code == 200:
    soup = BeautifulSoup(res.content, "html.parser")
    textos.append({"text": soup.get_text(separator=' ', strip=True), "source": "repositorio.unal.edu.co"})
print(f"Se cargaron {len(textos)} fuentes de texto.")


In [ ]:
chunks = []
chunk_size = 800
chunk_overlap = 100
for doc in textos:
    text = doc["text"]
    source = doc["source"]
    for i in range(0, len(text), chunk_size - chunk_overlap):
        chunk_str = text[i:i+chunk_size]
        if len(chunk_str.strip()) > 50:
            chunks.append({"text": chunk_str, "source": source})
print(f"{len(chunks)} fragmentos generados.")


In [ ]:
print("Generando embeddings con Google API...")
embeddings = []
def get_embedding(text):
    return genai.embed_content(model="models/embedding-001", content=text, task_type="retrieval_document")["embedding"]

for chunk in chunks:
    embeddings.append(get_embedding(chunk["text"]))

d = len(embeddings[0])
index = faiss.IndexFlatL2(d)
index.add(np.array(embeddings, dtype=np.float32))
print("Base FAISS construida.")


In [ ]:
from duckduckgo_search import DDGS

def base_conocimientos_unal(query: str) -> str:
    """Busca información específica sobre la Universidad Nacional, tesis, documentos locales y tu base de datos prioritaria."""
    q_emb = genai.embed_content(model="models/embedding-001", content=query, task_type="retrieval_query")["embedding"]
    D, I = index.search(np.array([q_emb], dtype=np.float32), 3)
    return "

".join([chunks[idx]['text'] for idx in I[0]])

def busqueda_web(query: str) -> str:
    """Busca información reciente o de cultura general en internet abierto."""
    results = DDGS().text(query, max_results=3)
    try:
         return "
".join([f"{r.get('title')}: {r.get('body')}" for r in results])
    except:
         return "Sin resultados."

# El SDK Gemini 1.5 es capaz de llamar automáticamente a varias funciones (Function Calling)
model = genai.GenerativeModel(
    'gemini-1.5-flash',
    tools=[base_conocimientos_unal, busqueda_web],
    system_instruction="Eres un asistente avanzado que puede buscar en internet usando 'busqueda_web' o buscar en documentos sobre la Universidad usando 'base_conocimientos_unal'."
)
chat = model.start_chat(enable_automatic_function_calling=True)


In [ ]:
def responder_agente(mensaje, historial):
    try:
        # El chat ya maneja las idas y venidas del tool calling de forma transparente
        res = chat.send_message(mensaje)
        return res.text
    except Exception as e:
        return f"Ocurrió un error en el razonamiento del agente: {e}"

gr.ChatInterface(responder_agente, title="Agente Tool-Calling Automático (Nativo Gemini SDK)").launch(share=True)
